In [1]:
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay, r2_score, root_mean_squared_error, mean_absolute_error, balanced_accuracy_score, RocCurveDisplay, PrecisionRecallDisplay, make_scorer, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from utils import load_plastchem, calculate_mfps, focal_binary_loss
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.combine import SMOTEENN
from sklearn.decomposition import PCA
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import StratifiedKFold, HalvingGridSearchCV, GridSearchCV
from xgboost import XGBClassifier
import pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler
import numpy as np
import joblib

In [2]:
# Data preparation
plast_chem, pcc = load_plastchem(Path.cwd() / "data" / "plastchem_db_v1.0.tsv")

hazard_cols = [
        'Hazard_information_Carcinogenic', 'Hazard_information_Mutagenic', 'Hazard_information_Reproduction', 'Hazard_information_CMR', 'Hazard_information_STOT',
        'Hazard_information_EDC', 'Hazard_information_Aquatic_toxicity', 'Hazard_information_PBT', 'Hazard_information_vPvB', 'Hazard_information_PMT', 'Hazard_information_vPvM'
    ]

# exclude NAs
plast_chem_hazard = plast_chem[plast_chem[hazard_cols].notna().any(axis=1)]
plast_chem_hazard = plast_chem_hazard.where(plast_chem_hazard['Hazard_information_Toxicity_score'] != 0.25)

print(plast_chem_hazard['Hazard_information_Hazard_score'].value_counts())

# fingerprints
fps = calculate_mfps(plast_chem_hazard, 'Properties_isomeric_smiles')

mask = [fp is not None for fp in fps]
plast_chem_hazard = plast_chem_hazard[mask].copy()
fps = [fp for fp in fps if fp is not None]

X_fps = np.vstack(fps)

Hazard_information_Hazard_score
2.0    3049
0.5    1188
1.0    1041
0.0     161
Name: count, dtype: int64


[17:21:01] WARNING: not removing hydrogen atom without neighbors
[17:21:01] WARNING: not removing hydrogen atom without neighbors
[17:21:01] WARNING: not removing hydrogen atom without neighbors
[17:21:01] WARNING: not removing hydrogen atom without neighbors
[17:21:01] WARNING: not removing hydrogen atom without neighbors
[17:21:01] WARNING: not removing hydrogen atom without neighbors


[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] Explicit valence for atom # 1 Cl, 3, is greater than permitted
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors


[17:21:02] WARNING: not removing hydrogen atom without neighbors
[17:21:02] WARNING: not removing hydrogen atom without neighbors


In [3]:
# include Properties into Data, not just fingerprints
prop_cols = ['Properties_molecular_weight', 'Properties_tpsa', 'Properties_complexity']
props = plast_chem_hazard[prop_cols].astype(float)

# NaNs mit mean interpoliert
props_scaled = StandardScaler().fit_transform(props.fillna(props.mean()))

X = np.hstack([X_fps, props_scaled]) 

print(X)

[[ 0.          0.          0.         ... -0.52817419 -0.4741337
  -0.35921187]
 [ 0.          0.          0.         ... -0.51378725 -0.02963713
  -0.33760701]
 [ 0.          0.          0.         ... -0.33211119 -0.29735985
  -0.2463865 ]
 ...
 [ 0.          1.          0.         ... -0.36398099 -0.35628446
  -0.30159891]
 [ 0.          0.          0.         ... -0.59005624 -0.73417059
  -0.40962321]
 [ 0.          1.          0.         ...  0.86174971 -0.06038041
   0.58900137]]


### Data preparation 
- Merging hazard categories
- Splitting

In [4]:
# less hazardous and hazardous 1 and 2 get same category
y = plast_chem_hazard['Hazard_information_Hazard_score'].replace(0.5,1).replace(2,1).replace(3,1)
X = X.copy()
print(y.value_counts(normalize=True))

Hazard_information_Hazard_score
1.0    0.967144
0.0    0.032856
Name: proportion, dtype: float64


In [5]:
# splitting
X_exploration, X_test, y_exploration, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

### Training with Resampling

In [6]:
%%time
pipeline = Pipeline([
    # PCA for dimensionality reduction
    ("pca", PCA(random_state=0)),
    # combined oversampling and undersampling using SMOTE + ENN
    ("smoteenn", SMOTEENN(random_state=0)),
    # train xgboost
    ("clf", XGBClassifier(tree_method="hist",
                          random_state=0,
                          n_jobs=-1,
                          max_delta_step=3,
                          objective=focal_binary_loss))
])

param_grid = {
    "pca__n_components": [160, 190, 220],
    "smoteenn__sampling_strategy": [0.1, 0.2, 0.3],
    "clf__max_depth": [3, 5, 7],
    "clf__learning_rate": [0.001, 0.01, 0.1],
    "clf__n_estimators": [100, 200, 300],
    "clf__colsample_bylevel": [0.5, 0.75, 1.0]
}

# inner/outer cv for nested cv
inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=1
)

# set n_jobs to 1 here, let xgboost handle parallelization
# halving grid search not possible because data during PCA is not enough
search = GridSearchCV(
    pipeline,
    param_grid,
    cv=inner_cv,
    scoring="balanced_accuracy",
    verbose=10,
    n_jobs=-1,
    error_score="raise"
)

nested_scores = cross_val_score(
    search,
    X_exploration,
    y_exploration,
    cv=outer_cv,
    scoring="balanced_accuracy",
    n_jobs=1
)

print("Nested CV scores:", nested_scores)
print("Mean balanced accuracy:", nested_scores.mean())
print("Std:", nested_scores.std())

Fitting 5 folds for each of 729 candidates, totalling 3645 fits


Fitting 5 folds for each of 729 candidates, totalling 3645 fits


Fitting 5 folds for each of 729 candidates, totalling 3645 fits


Fitting 5 folds for each of 729 candidates, totalling 3645 fits


Fitting 5 folds for each of 729 candidates, totalling 3645 fits


Nested CV scores: [0.59808426 0.66566521 0.56273858 0.45382508 0.57781082]
Mean balanced accuracy: 0.5716247895067971
Std: 0.0686115264570823
CPU times: user 17min 19s, sys: 14.9 s, total: 17min 34s
Wall time: 38min 37s


In [7]:
%%time
search.fit(X_exploration, y_exploration)

Fitting 5 folds for each of 729 candidates, totalling 3645 fits


[CV 2/5; 4/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=190, smoteenn__sampling_strategy=0.1
[CV 2/5; 4/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=190, smoteenn__sampling_strategy=0.1;, score=0.500 total time=   2.9s
[CV 5/5; 17/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.2
[CV 5/5; 17/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.2;, score=0.552 total time=   3.8s
[CV 4/5; 30/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.3
[CV 4/5; 30/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

[CV 5/5; 1/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.1
[CV 5/5; 1/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.1;, score=0.500 total time=   2.7s
[CV 2/5; 14/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=190, smoteenn__sampling_strategy=0.2
[CV 2/5; 14/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=190, smoteenn__sampling_strategy=0.2;, score=0.567 total time=   2.8s
[CV 4/5; 26/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=220, smoteenn__sampling_strategy=0.2
[CV 4/5; 26/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

[CV 2/5; 8/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.2
[CV 2/5; 8/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.2;, score=0.544 total time=   3.0s
[CV 5/5; 18/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.3
[CV 5/5; 18/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.3;, score=0.525 total time=   4.1s
[CV 5/5; 31/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=190, smoteenn__sampling_strategy=0.1
[CV 5/5; 31/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

[CV 4/5; 11/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.2
[CV 4/5; 11/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.2;, score=0.608 total time=   3.5s
[CV 1/5; 24/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=190, smoteenn__sampling_strategy=0.3
[CV 1/5; 24/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=190, smoteenn__sampling_strategy=0.3;, score=0.566 total time=   5.1s
[CV 4/5; 37/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.1
[CV 4/5; 37/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001

...

[CV 2/5; 10/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.1
[CV 2/5; 10/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.1;, score=0.500 total time=   3.4s
[CV 4/5; 23/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=190, smoteenn__sampling_strategy=0.2
[CV 4/5; 23/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=190, smoteenn__sampling_strategy=0.2;, score=0.639 total time=   5.0s
[CV 5/5; 36/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.3
[CV 5/5; 36/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001

[CV 5/5; 10/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.1
[CV 5/5; 10/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.1;, score=0.500 total time=   3.6s
[CV 2/5; 24/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=190, smoteenn__sampling_strategy=0.3
[CV 2/5; 24/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=190, smoteenn__sampling_strategy=0.3;, score=0.616 total time=   5.1s
[CV 5/5; 37/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.1
[CV 5/5; 37/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001

[CV 2/5; 2/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.2
[CV 2/5; 2/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.2;, score=0.592 total time=   2.8s
[CV 5/5; 15/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=190, smoteenn__sampling_strategy=0.3
[CV 5/5; 15/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=190, smoteenn__sampling_strategy=0.3;, score=0.593 total time=   3.5s
[CV 1/5; 28/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.1
[CV 1/5; 28/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

CPU times: user 6min 5s, sys: 3.36 s, total: 6min 8s
Wall time: 8min 51s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__colsample_bylevel': [0.5, 0.75, ...], 'clf__learning_rate': [0.001, 0.01, ...], 'clf__max_depth': [3, 5, ...], 'clf__n_estimators': [100, 200, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'balanced_accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",10
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_est

In [8]:
print("Best parameter (CV score=%0.3f):" % search.best_score_)
print(search.best_params_)

Best parameter (CV score=0.616):
{'clf__colsample_bylevel': 1.0, 'clf__learning_rate': 0.01, 'clf__max_depth': 7, 'clf__n_estimators': 300, 'pca__n_components': 160, 'smoteenn__sampling_strategy': 0.3}


### Training without Resampling

In [9]:
nr_pipeline = Pipeline([
    # PCA for dimensionality reduction
    ("pca", PCA(random_state=0)),
    # train xgboost
    ("clf", XGBClassifier(tree_method="hist",
                          random_state=0,
                          n_jobs=1,
                          max_delta_step=3,
                          objective=focal_binary_loss))
])

nr_param_grid = {
    "pca__n_components": [160, 190, 220],
    "clf__max_depth": [3, 7],
    "clf__learning_rate": [0.01, 0.1],
    "clf__n_estimators": [100, 200, 300],
    "clf__colsample_bylevel": [0.5, 0.75, 1.0]
}

# inner/outer cv for nested cv
nr_inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)

nr_outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=1
)

nr_search = GridSearchCV(
    nr_pipeline,
    nr_param_grid,
    cv=nr_inner_cv,
    scoring="balanced_accuracy",
    verbose=10,
    n_jobs=-1,
    error_score="raise"
)

nr_nested_scores = cross_val_score(
    nr_search,
    X_exploration,
    y_exploration,
    cv=nr_outer_cv,
    scoring="balanced_accuracy",
    n_jobs=1
)

print("Nested CV scores:", nr_nested_scores)
print("Mean balanced accuracy:", nr_nested_scores.mean())
print("Std:", nr_nested_scores.std())

Fitting 5 folds for each of 108 candidates, totalling 540 fits


[CV 2/5; 12/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.3
[CV 2/5; 12/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.3;, score=0.619 total time=   3.3s
[CV 3/5; 24/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=190, smoteenn__sampling_strategy=0.3
[CV 3/5; 24/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=190, smoteenn__sampling_strategy=0.3;, score=0.563 total time=   5.1s
[CV 2/5; 38/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=200, pca__n_components=160, smoteenn__sampling_strategy=0.2
[CV 2/5; 38/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001

[CV 4/5; 2/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.2
[CV 4/5; 2/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.2;, score=0.608 total time=   2.8s
[CV 3/5; 15/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=190, smoteenn__sampling_strategy=0.3
[CV 3/5; 15/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=190, smoteenn__sampling_strategy=0.3;, score=0.562 total time=   4.2s
[CV 1/5; 31/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=190, smoteenn__sampling_strategy=0.1
[CV 1/5; 31/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

[CV 3/5; 8/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.2
[CV 3/5; 8/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.2;, score=0.541 total time=   2.9s
[CV 4/5; 21/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=160, smoteenn__sampling_strategy=0.3
[CV 4/5; 21/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=160, smoteenn__sampling_strategy=0.3;, score=0.635 total time=   4.1s
[CV 2/5; 33/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=190, smoteenn__sampling_strategy=0.3
[CV 2/5; 33/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

[CV 1/5; 5/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=190, smoteenn__sampling_strategy=0.2
[CV 1/5; 5/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=190, smoteenn__sampling_strategy=0.2;, score=0.554 total time=   3.1s
[CV 1/5; 21/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=160, smoteenn__sampling_strategy=0.3
[CV 1/5; 21/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=300, pca__n_components=160, smoteenn__sampling_strategy=0.3;, score=0.569 total time=   4.5s
[CV 2/5; 34/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.1
[CV 2/5; 34/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

...

[CV 2/5; 9/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.3
[CV 2/5; 9/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.3;, score=0.588 total time=   2.9s
[CV 2/5; 18/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.3
[CV 2/5; 18/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.3;, score=0.617 total time=   3.6s
[CV 5/5; 29/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.2
[CV 5/5; 29/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

[CV 5/5; 9/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.3
[CV 5/5; 9/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.3;, score=0.547 total time=   2.9s
[CV 1/5; 17/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.2
[CV 1/5; 17/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.2;, score=0.551 total time=   3.6s
[CV 3/5; 29/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.2
[CV 3/5; 29/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

Fitting 5 folds for each of 108 candidates, totalling 540 fits


[CV 5/5; 7/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.1
[CV 5/5; 7/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.1;, score=0.500 total time=   2.8s
[CV 2/5; 17/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.2
[CV 2/5; 17/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.2;, score=0.544 total time=   3.5s
[CV 2/5; 29/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.2
[CV 2/5; 29/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

Nested CV scores: [0.4992224  0.54312173 0.51961685 0.4992224  0.49922118]
Mean balanced accuracy: 0.5120809114384789
Std: 0.01741481177483541


In [10]:
%%time
nr_search.fit(X_exploration, y_exploration)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


[CV 4/5; 7/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.1
[CV 4/5; 7/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.1;, score=0.500 total time=   2.8s
[CV 1/5; 16/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.1
[CV 1/5; 16/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.1;, score=0.527 total time=   3.7s
[CV 4/5; 29/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.2
[CV 4/5; 29/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

[CV 3/5; 9/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.3
[CV 3/5; 9/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=100, pca__n_components=220, smoteenn__sampling_strategy=0.3;, score=0.537 total time=   2.9s
[CV 4/5; 17/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.2
[CV 4/5; 17/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=3, clf__n_estimators=200, pca__n_components=220, smoteenn__sampling_strategy=0.2;, score=0.637 total time=   3.9s
[CV 5/5; 30/729] START clf__colsample_bylevel=0.5, clf__learning_rate=0.001, clf__max_depth=5, clf__n_estimators=100, pca__n_components=160, smoteenn__sampling_strategy=0.3
[CV 5/5; 30/729] END clf__colsample_bylevel=0.5, clf__learning_rate=0.001, 

CPU times: user 41 s, sys: 333 ms, total: 41.3 s
Wall time: 1min 12s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__colsample_bylevel': [0.5, 0.75, ...], 'clf__learning_rate': [0.01, 0.1], 'clf__max_depth': [3, 7], 'clf__n_estimators': [100, 200, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'balanced_accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",10
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``att

In [11]:
print("Best parameter no resampling (CV score=%0.3f):" % nr_search.best_score_)
print(nr_search.best_params_)

Best parameter no resampling (CV score=0.522):
{'clf__colsample_bylevel': 1.0, 'clf__learning_rate': 0.01, 'clf__max_depth': 7, 'clf__n_estimators': 100, 'pca__n_components': 160}


### PCA measure + std/mean test score

In [12]:
i = search.best_index_
best_model = search.best_estimator_
print(search.cv_results_['std_test_score'][i], search.cv_results_['mean_test_score'][i])
pca = best_model.named_steps["pca"]
print(pca.explained_variance_ratio_.sum())

0.03313682626303311 0.6160691217427312
0.5921566149534919


In [13]:
i_nr = nr_search.best_index_
nr_best_model = nr_search.best_estimator_
print(nr_search.cv_results_['std_test_score'][i_nr], nr_search.cv_results_['mean_test_score'][i_nr])
pca_nr = nr_best_model.named_steps["pca"]
print(pca_nr.explained_variance_ratio_.sum())

0.014159567202895804 0.5218550759763824
0.5921566149534919


### Baseline (Dummy Classifier)

In [14]:
dummy = DummyClassifier(strategy='most_frequent', random_state=0)
dummy.fit(X_exploration, y_exploration)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.03,0.97]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[float64](2,)","[0.,1.]"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,2051
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [15]:
# make predictions
dummy_preds = dummy.predict(X_test)
dummy_probs = dummy.predict_proba(X_test)
print(balanced_accuracy_score(y_test, dummy_preds))
print(classification_report(y_test, dummy_preds))
print(roc_auc_score(y_test, dummy_probs[:, 1]))

0.5
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        47
         1.0       0.97      1.00      0.98      1378

    accuracy                           0.97      1425
   macro avg       0.48      0.50      0.49      1425
weighted avg       0.94      0.97      0.95      1425

0.5


/vast/ki64cah/.conda/envs/plastchem/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/vast/ki64cah/.conda/envs/plastchem/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/vast/ki64cah/.conda/envs/plastchem/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

### Test

In [16]:
test_preds = best_model.predict(X_test)
test_probs = best_model.predict_proba(X_test)
print(balanced_accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))
print(roc_auc_score(y_test, test_probs[:, 1]))

0.6158015007874502
              precision    recall  f1-score   support

         0.0       0.17      0.28      0.21        47
         1.0       0.97      0.96      0.96      1378

    accuracy                           0.93      1425
   macro avg       0.57      0.62      0.59      1425
weighted avg       0.95      0.93      0.94      1425

0.7233965352190964


In [17]:
nr_test_preds = nr_best_model.predict(X_test)
nr_test_probs = nr_best_model.predict_proba(X_test)
print(balanced_accuracy_score(y_test, nr_test_preds))
print(classification_report(y_test, nr_test_preds))
print(roc_auc_score(y_test, nr_test_probs[:, 1]))

0.49963715529753266
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        47
         1.0       0.97      1.00      0.98      1378

    accuracy                           0.97      1425
   macro avg       0.48      0.50      0.49      1425
weighted avg       0.94      0.97      0.95      1425

0.6904625883951456


### Saving for visualizations

In [18]:
# save all predictions from final run for visualisations on local machine
# for run with resampling
np.savez(Path.cwd() / "data" / "results_test.npz",
         y_test = y_test, X_test = X_test, test_preds = test_preds, test_probs = test_probs,
         dummy_preds = dummy_preds, dummy_probs = dummy_probs,
         X_exploration=X_exploration, y_exploration=y_exploration,
         nested_scores=nested_scores)
joblib.dump(search, Path.cwd() / "data" / "search.joblib")
joblib.dump(best_model, Path.cwd() / "data" / "best_model.joblib")

['/work/ki64cah/plast-chem/analysis/data/best_model.joblib']

In [19]:
# and run without resampling
np.savez(Path.cwd() / "data" / "results_test_no_resample.npz",
         nr_test_preds = nr_test_preds, nr_test_probs = nr_test_probs,
         nr_nested_scores = nr_nested_scores)
joblib.dump(nr_search, Path.cwd() / "data" / "nr_search.joblib")
joblib.dump(nr_best_model, Path.cwd() / "data" / "nr_best_model.joblib")

['/work/ki64cah/plast-chem/analysis/data/nr_best_model.joblib']